# 第12章 自然言語処理の基礎

『Python機械学習スタートブック』のコードをGoogle Colabで実行するためのノートブックです。
コードは書籍のリスト番号順に並んでいます。上から順に実行してください。

- 解説（Web教材）: https://ml.kano.ac/chapters/12/
- 演習の解答: https://ml.kano.ac/solutions/12/

## テキストデータの基礎

### Pythonにおける文字列の基本操作

**リスト 12.1**　`join`・`split`・`replace`による文字列操作

In [ ]:
words = ["自然", "言語", "処理"]
print(" ".join(words))           # 自然 言語 処理

text = "Python は 自然言語処理 に 便利 です"
print(text.split(" "))           # ["Python", "は", "自然言語処理", ...]

s = "今日は天気が良いです。明日も天気が良いでしょう。"
print(s.replace("天気が良い", "晴れ"))  # 今日は晴れです。明日も晴れでしょう。

### ファイルの読み込み

**リスト 12.2**　テキストファイルの作成

In [ ]:
# サンプルファイルの作成
text = "自然言語処理は人工知能の一分野です。\nテキストデータを分析する技術です。\n"
with open("sample.txt", "w", encoding="utf-8") as f:
    f.write(text)

**リスト 12.3**　テキストファイルの読み込み

In [ ]:
# テキストファイルの読み込み
with open("sample.txt", "r", encoding="utf-8") as f:
    text = f.read()
print(text)

**リスト 12.4**　`readlines()`による行ごとの読み込み

In [ ]:
with open("sample.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()
print(lines)

### 日本語テキストの特徴

**リスト 12.5**　英語と日本語の単語分割の違い

In [ ]:
# 英語: スペースで単語に分割できる
english = "Natural language processing is important"
print(english.split(" "))

# 日本語: スペースがないため単純に分割できない
japanese = "自然言語処理は重要です"
print(japanese.split(" "))

## 形態素解析

### MeCabのインストール

**リスト 12.6**　MeCabのインストール

In [ ]:
# Google Colabでの実行
!pip install mecab-python3 unidic-lite

### MeCabの基本的な使い方

#### parse()による解析

**リスト 12.7**　`parse()`による形態素解析

In [ ]:
import MeCab

tagger = MeCab.Tagger()
text = "自然言語処理を学びます"
result = tagger.parse(text)
print(result)

#### -Owakatiオプション（分かち書き）

**リスト 12.8**　`-Owakati`オプションによる分かち書き

In [ ]:
import MeCab

tagger = MeCab.Tagger("-Owakati")
text = "自然言語処理を学びます"
result = tagger.parse(text)
print(result)

### parseToNode()による品詞情報の取得

**リスト 12.9**　`parseToNode()`による品詞情報の取得

In [ ]:
import MeCab

tagger = MeCab.Tagger()
text = "自然言語処理を学びます"

node = tagger.parseToNode(text)
while node:
    # 表層形（surface）と素性情報（feature）
    if node.surface != "":
        word = node.surface
        pos = node.feature.split(",")[0]  # 品詞の取得
        print(f"{word}\t{pos}")
    node = node.next

## テキストの前処理

### テキストクリーニング

#### replaceによるクリーニング

**リスト 12.10**　`replace()`によるテキストクリーニング

In [ ]:
text = "今日の\t天気は<b>晴れ</b>です。\n"
# タブ文字の除去
text = text.replace("\t", "")
# 改行文字の除去
text = text.replace("\n", "")
# HTMLタグの除去
text = text.replace("<b>", "")
text = text.replace("</b>", "")
print(text)

#### 正規表現によるクリーニング

**リスト 12.11**　正規表現によるテキストクリーニング

In [ ]:
import re

text = "今日の\t天気は<b>晴れ</b>です。\n"
# タブと改行を除去
text = re.sub(r"[\t\n]", "", text)
# HTMLタグを除去
text = re.sub(r"<.*?>", "", text)
print(text)

#### URL・メールアドレス・電話番号の除去

**リスト 12.12**　URL・メールアドレス・電話番号の除去

In [ ]:
import re

text = """
お問い合わせは https://example.com まで。
メール: info@example.com
電話: 03-1234-5678 または 090-1234-5678
"""

# URLの除去
text = re.sub(r"https?://[\w/:%#\$&\?\(\)~\.=\+\-]+", "", text)

# メールアドレスの除去
text = re.sub(r"[\w\.-]+@[\w\.-]+\.\w+", "", text)

# 電話番号の除去（ハイフン区切り）
text = re.sub(r"\d{2,4}-\d{2,4}-\d{4}", "", text)

print(text)

**リスト 12.13**　余分な空白の整理

In [ ]:
import re

text = "  これは  テスト  です  "
# 連続する空白を1つに置換
text = re.sub(r"\s+", " ", text)
# 前後の空白を除去
text = text.strip()
print(text)

### テキスト正規化

#### ストップワードの除去

**リスト 12.14**　ストップワードの除去

In [ ]:
import MeCab

# ストップワードの定義
stop_words = ["は", "の", "が", "を", "に", "へ", "と", "で", "た",
              "です", "ます", "する", "いる", "ある", "こと", "もの",
              "それ", "これ", "ない", "なる", "れる", "られる"]

tagger = MeCab.Tagger("-Owakati")
text = "自然言語処理は人工知能の重要な分野です"
words = tagger.parse(text).strip().split(" ")
print("除去前:", words)

# ストップワードの除去
filtered = [w for w in words if w not in stop_words]
print("除去後:", filtered)

#### 品詞によるフィルタリング

**リスト 12.15**　品詞によるフィルタリング

In [ ]:
import MeCab

tagger = MeCab.Tagger()
text = "機械学習の技術は急速に進歩している"

# 名詞、動詞、形容詞のみを抽出
target_pos = ["名詞", "動詞", "形容詞"]
words = []

node = tagger.parseToNode(text)
while node:
    if node.surface != "":
        pos = node.feature.split(",")[0]
        if pos in target_pos:
            words.append(node.surface)
    node = node.next

print(words)

### 前処理パイプライン

**リスト 12.16**　前処理パイプラインの関数化

In [ ]:
import re
import MeCab

# テキストの前処理パイプライン
def preprocess(text):
    # 1. テキストクリーニング
    text = re.sub(r"https?://[\w/:%#\$&\?\(\)~\.=\+\-]+", "", text)  # URL除去
    text = re.sub(r"[\w\.-]+@[\w\.-]+\.\w+", "", text)  # メールアドレス除去
    text = re.sub(r"\d{2,4}-\d{2,4}-\d{4}", "", text)  # 電話番号除去
    text = re.sub(r"[!-/:-@\[-`{-~]", "", text)  # 半角記号除去
    text = re.sub(r"[！-／：-＠［-｀｛-～、。・「」『』【】]", "", text)  # 全角記号除去
    text = re.sub(r"\s+", " ", text).strip()  # 空白の正規化

    # 2. 形態素解析と品詞フィルタリング
    tagger = MeCab.Tagger()
    target_pos = ["名詞", "動詞", "形容詞"]
    words = []

    node = tagger.parseToNode(text)
    while node:
        if node.surface != "":
            pos = node.feature.split(",")[0]
            if pos in target_pos:
                words.append(node.surface)
        node = node.next

    return words

**リスト 12.17**　前処理パイプラインの実行

In [ ]:
text = "詳細は https://example.com をご覧ください！お問い合わせ: info@test.com"
result = preprocess(text)
print(result)

### 複数文書の前処理

**リスト 12.18**　複数文書への前処理の適用

In [ ]:
documents = [
    "自然言語処理は人工知能の一分野です。",
    "機械学習を使ってテキストを分析します。",
    "深層学習の発展により精度が向上しました。",
]

processed_docs = []
for doc in documents:
    words = preprocess(doc)
    processed_docs.append(" ".join(words))

for i, doc in enumerate(processed_docs):
    print(f"文書{i+1}: {doc}")

## テキストのベクトル化（BoW、TF-IDF）

### CountVectorizerによる実装

**リスト 12.19**　`CountVectorizer`によるBoWベクトル化

In [ ]:
import MeCab
from sklearn.feature_extraction.text import CountVectorizer

# サンプル文書
documents = [
    "自然言語処理は人工知能の一分野です",
    "機械学習で自然言語を処理します",
    "深層学習は機械学習の一手法です",
]

# MeCabで分かち書き
tagger = MeCab.Tagger("-Owakati")
wakati_docs = [tagger.parse(doc).strip() for doc in documents]
print("分かち書き結果:")
for i, doc in enumerate(wakati_docs):
    print(f"  文書{i+1}: {doc}")

# CountVectorizerでBoWベクトルを生成
vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(wakati_docs)

# 語彙の確認
print("\n語彙:", vectorizer.get_feature_names_out())

# BoW行列の表示
print("\nBoW行列:")
print(bow_matrix.toarray())

### fit, transform, fit_transformの違い

**リスト 12.20**　`fit_transform()`と`transform()`の使い分け

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# 訓練データ
train_docs = ["猫 は かわいい", "犬 も かわいい"]

# テストデータ
test_docs = ["猫 と 犬 は 友達"]

# 訓練データで学習＋変換（fit_transform）
# token_pattern で1文字の単語も語彙に含める
vectorizer = CountVectorizer(token_pattern=r"(?u)\b\w+\b")
train_matrix = vectorizer.fit_transform(train_docs)
print("学習した語彙:", vectorizer.get_feature_names_out())
print("訓練データのベクトル:")
print(train_matrix.toarray())

# テストデータには transform のみ適用
test_matrix = vectorizer.transform(test_docs)
print("\nテストデータのベクトル:")
print(test_matrix.toarray())

### CountVectorizerのパラメータ

**リスト 12.21**　`CountVectorizer`の主なパラメータ

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = ["猫 は かわいい 動物", "犬 も かわいい 動物", "猫 も 犬 も かわいい"]
tp = r"(?u)\b\w+\b"  # 1文字の単語も語彙に含める設定

# max_df: 出現頻度が高すぎる単語を除外（全文書の80%以上に出現する単語を除外）
vec1 = CountVectorizer(max_df=0.8, token_pattern=tp)
result1 = vec1.fit_transform(docs)
print("max_df=0.8 の語彙:", vec1.get_feature_names_out())

# min_df: 出現頻度が低すぎる単語を除外（2文書未満にしか出現しない単語を除外）
vec2 = CountVectorizer(min_df=2, token_pattern=tp)
result2 = vec2.fit_transform(docs)
print("min_df=2 の語彙:", vec2.get_feature_names_out())

# max_features: 語彙の最大数を指定（上位3語のみ）
vec3 = CountVectorizer(max_features=3, token_pattern=tp)
result3 = vec3.fit_transform(docs)
print("max_features=3 の語彙:", vec3.get_feature_names_out())

### N-gram

**リスト 12.22**　`ngram_range`によるN-gramの生成

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

docs = ["自然 言語 処理 を 学ぶ"]
tp = r"(?u)\b\w+\b"  # 1文字の単語も語彙に含める設定

# ユニグラムのみ（デフォルト）
vec_uni = CountVectorizer(token_pattern=tp, ngram_range=(1, 1))
vec_uni.fit(docs)
print("ユニグラム:", vec_uni.get_feature_names_out())

# バイグラムのみ
vec_bi = CountVectorizer(token_pattern=tp, ngram_range=(2, 2))
vec_bi.fit(docs)
print("バイグラム:", vec_bi.get_feature_names_out())

# ユニグラム + バイグラム
vec_both = CountVectorizer(token_pattern=tp, ngram_range=(1, 2))
vec_both.fit(docs)
print("ユニグラム+バイグラム:", vec_both.get_feature_names_out())

### TfidfVectorizerによる実装

**リスト 12.23**　`TfidfVectorizer`によるTF-IDFベクトル化

In [ ]:
import MeCab
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# サンプル文書
documents = [
    "自然言語処理は人工知能の一分野です",
    "機械学習で自然言語を処理します",
    "深層学習も自然言語を処理します",
]

# MeCabで分かち書き
tagger = MeCab.Tagger("-Owakati")
wakati_docs = [tagger.parse(doc).strip() for doc in documents]

# TfidfVectorizerでTF-IDFベクトルを生成
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(wakati_docs)

# 結果をDataFrameで見やすく表示
feature_names = tfidf_vectorizer.get_feature_names_out()
df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names,
    index=["文書1", "文書2", "文書3"]
)
print(df.round(2))

### BoWとTF-IDFの比較

**リスト 12.24**　BoWとTF-IDFの比較

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

documents = [
    "猫 は かわいい 動物",
    "犬 も かわいい 動物",
    "猫 は 自由 な 動物",
]
tp = r"(?u)\b\w+\b"  # 1文字の単語も語彙に含める設定

# Bag-of-Words
count_vec = CountVectorizer(token_pattern=tp)
bow = count_vec.fit_transform(documents)
print("【Bag-of-Words】")
df_bow = pd.DataFrame(bow.toarray(), columns=count_vec.get_feature_names_out(),
                       index=["文書1", "文書2", "文書3"])
print(df_bow)

# TF-IDF
tfidf_vec = TfidfVectorizer(token_pattern=tp)
tfidf = tfidf_vec.fit_transform(documents)
print("\n【TF-IDF】")
df_tfidf = pd.DataFrame(tfidf.toarray(), columns=tfidf_vec.get_feature_names_out(),
                          index=["文書1", "文書2", "文書3"])
print(df_tfidf.round(2))

## 類似文書検索

### TF-IDFベクトルを用いた文書間の類似度計算

**リスト 12.25**　文書間のコサイン類似度の計算

In [ ]:
import MeCab
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# サンプル文書コーパス
documents = [
    "機械学習はデータからパターンを学習する技術です",
    "深層学習はニューラルネットワークを用いた機械学習の手法です",
    "自然言語処理はテキストデータを分析する技術です",
    "画像認識は写真や動画から物体を検出する技術です",
    "機械学習を使ってテキストを分類することができます",
]

# MeCabで分かち書き
tagger = MeCab.Tagger("-Owakati")
wakati_docs = [tagger.parse(doc).strip() for doc in documents]

# TF-IDFベクトルの生成
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(wakati_docs)

# 全文書間のコサイン類似度を計算
cos_sim = cosine_similarity(tfidf_matrix)

# 結果の表示
labels = [f"文書{i+1}" for i in range(len(documents))]
df_sim = pd.DataFrame(cos_sim, index=labels, columns=labels)
print(df_sim.round(2))

### 類似文書の検索と順位付け

**リスト 12.26**　類似文書の検索と順位付け

In [ ]:
import MeCab
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# クエリに類似した文書を検索して順位付けする
def search_similar(query, documents, vectorizer, tfidf_matrix, top_n=3):
    # クエリをMeCabで分かち書き
    tagger = MeCab.Tagger("-Owakati")
    query_wakati = tagger.parse(query).strip()

    # クエリをTF-IDFベクトルに変換（学習済みのvectorizerを使用）
    query_vec = vectorizer.transform([query_wakati])

    # クエリとコーパス内の各文書とのコサイン類似度を計算
    similarities = cosine_similarity(query_vec, tfidf_matrix)[0]

    # 類似度の高い順にソート
    ranking = np.argsort(similarities)[::-1]

    print(f"クエリ: 「{query}」\n")
    print("類似文書ランキング:")
    for rank, idx in enumerate(ranking[:top_n], 1):
        print(f"  {rank}位 (類似度: {similarities[idx]:.4f}): {documents[idx]}")

# 検索の実行
query = "テキストデータの分析と分類の方法"
search_similar(query, documents, vectorizer, tfidf_matrix)

## 演習問題

### 演習 12-1: テキストクリーニング

以下のテキストには URL・メールアドレス・電話番号・装飾記号が含まれています。正規表現を使ってこれらをすべて除去し、最後に余分な空白も整えてください。

In [ ]:
text = """
明日のセミナー【締切間近！】の案内です★
詳しくは https://example.com まで！
質問は contact@example.jp までお願いします。
電話番号： 03-1234-5678
"""

**タスク**

1. `re.sub` で URL（`https?://...`）を削除する
2. `re.sub` でメールアドレス（`\S+@\S+`）を削除する
3. `re.sub` で電話番号（`\d{2,4}-\d{2,4}-\d{4}`）を削除する
4. `re.sub` で装飾記号（`【`・`】`・`★`）を削除する
5. `re.sub` で連続する空白を 1 つにまとめ、`strip()` で前後の空白を除去する

In [ ]:
import re

text = """
明日のセミナー【締切間近！】の案内です★
詳しくは https://example.com まで！
質問は contact@example.jp までお願いします。
電話番号： 03-1234-5678
"""

# 1. URL の削除

# 2. メールアドレスの削除

# 3. 電話番号の削除

# 4. 装飾記号の削除

# 5. 余分な空白の整理 + strip

print(text)

[解答例を見る](https://ml.kano.ac/solutions/12/#solution-12-1)

### 演習 12-2: 名詞（内容語）の抽出

本章の品詞フィルタリングのコードを変更し、指定した品詞のみを抽出する関数を作成して、以下の文から**名詞のみ**を抽出してください。

In [ ]:
text = "機械学習の技術は急速に進歩している"

**タスク**

1. `MeCab.Tagger()` で解析器を作成する
2. 指定した品詞のみを抽出する関数 `extract_by_pos(text, target_pos)` を定義する（`parseToNode()` で各ノードの品詞を調べ、`target_pos` に含まれる `surface` だけをリストに追加する）
3. 上の文から名詞のみを抽出して表示する
4. 名詞・動詞・形容詞を残した場合と結果を比較する
5. 自分で考えた文にも適用して、抽出結果を確認する

In [ ]:
import MeCab

text = "機械学習の技術は急速に進歩している"

# 1. MeCab のインスタンス生成

# 2. extract_by_pos(text, target_pos) 関数の定義

# 3. 名詞のみを抽出して表示

# 4. 名詞・動詞・形容詞を残した場合と比較

# 5. 自分で考えた文でも確認

[解答例を見る](https://ml.kano.ac/solutions/12/#solution-12-2)

### 演習 12-3: 前処理パイプラインの構築

複数の日本語テキストに対して、「クリーニング → 形態素解析 → 品詞フィルタ → ストップワード除去」までを一つの関数 `preprocess(text)` にまとめ、4 つのテキストを処理してください。

In [ ]:
texts = [
    "AI技術の発展は素晴らしいです！詳細は https://example.jp へ。",
    "機械学習を勉強することで、新しいことができるようになります。",
    "本日のテーマ：ディープラーニング入門。質問がある方は contact@ml.example.com まで★",
    "価格は1980円、送料は300円です。",
]
stopwords = {"する", "ある", "いる", "なる", "こと", "もの", "ため", "とき"}

**タスク**

1. 関数 `preprocess(text)` を定義する。中で以下を順に行う
    - URL（`https?://...`）とメールアドレスを `re.sub` で削除
    - 数字（`\d+`）を削除
    - 装飾記号（`【`・`】`・`★`・`：`）を削除
    - 連続する空白を整理
    - `MeCab` で形態素解析し、品詞が **名詞・動詞・形容詞** のものだけを残す
    - 上のストップワードに含まれる単語を除外する（`node.surface` での照合で OK）
2. 4 つのテキストそれぞれに `preprocess()` を適用し、結果を表示する

※ 本来ストップワードは活用形まで考慮した除去をすることが望ましいですが、本演習では surface（表層形）での照合で十分です。

In [ ]:
import re
import MeCab

texts = [
    "AI技術の発展は素晴らしいです！詳細は https://example.jp へ。",
    "機械学習を勉強することで、新しいことができるようになります。",
    "本日のテーマ：ディープラーニング入門。質問がある方は contact@ml.example.com まで★",
    "価格は1980円、送料は300円です。",
]
stopwords = {"する", "ある", "いる", "なる", "こと", "もの", "ため", "とき"}

# 1. preprocess(text) 関数の定義

# 2. 各テキストに適用して結果を表示

[解答例を見る](https://ml.kano.ac/solutions/12/#solution-12-3)

### 演習 12-4: BoW と TF-IDF の比較

本章のサンプル文書コーパス（5 文）に対して、`CountVectorizer`（BoW）と `TfidfVectorizer` のそれぞれでベクトル化を行い、同じ単語の値がどう異なるかを比較してください。

In [ ]:
documents = [
    "機械学習はデータからパターンを学習する技術です",
    "深層学習はニューラルネットワークを用いた機械学習の手法です",
    "自然言語処理はテキストデータを分析する技術です",
    "画像認識は写真や動画から物体を検出する技術です",
    "機械学習を使ってテキストを分類することができます",
]

**タスク**

1. `MeCab.Tagger("-Owakati")` で全文書を分かち書きする
2. `CountVectorizer` と `TfidfVectorizer` のそれぞれでベクトル化する
3. 「技術」「学習」「画像」「分類」などいくつかの単語について、文書ごとの BoW の値と TF-IDF の値を並べて表示する
4. 多くの文書に登場する単語と、特定の文書にしか登場しない単語とで、TF-IDF の値にどのような違いが現れるか考察する

In [ ]:
import MeCab
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

documents = [
    "機械学習はデータからパターンを学習する技術です",
    "深層学習はニューラルネットワークを用いた機械学習の手法です",
    "自然言語処理はテキストデータを分析する技術です",
    "画像認識は写真や動画から物体を検出する技術です",
    "機械学習を使ってテキストを分類することができます",
]

# 1. MeCab で分かち書き

# 2. CountVectorizer と TfidfVectorizer でベクトル化

# 3. 単語ごとに BoW と TF-IDF の値を並べて表示

なお、感情分析などのテキスト分類を題材とした演習は、第 13 章で扱います。

[解答例を見る](https://ml.kano.ac/solutions/12/#solution-12-4)